# SEGMENT 3 — Baseline Models

**Project**: HDB Resale Price Prediction  
**Models**: Linear Regression · KNN Regressor · Decision Tree Regressor  
**Primary Metrics**: MAE · RMSE · R²  

---

## Why Baseline Models First?
Before applying regularization or boosting, we need a **benchmark** to answer:
- How well does a simple linear relationship explain price? (Linear Regression)
- How well does local neighbourhood similarity explain price? (KNN)
- How well do threshold-based rules explain price? (Decision Tree)

These three models represent fundamentally different learning strategies and will reveal the **bias-variance landscape** of the problem.

---

## Key Metric Decisions for This Segment

| Metric | Formula | What it tells us |
|--------|---------|------------------|
| **MAE** | mean(\|y_pred - y_true\|) | Average dollar error; robust to outliers; easy to explain |
| **RMSE** | sqrt(mean((y_pred - y_true)²)) | Penalises large errors; competition standard |
| **R²** | 1 - SS_res/SS_tot | % of price variance explained; 1.0 = perfect |
| **CV RMSE** | 5-fold cross-val RMSE on train | Stability — does the score hold across data splits? |
| **Train-Test R² gap** | Train R² - Test R² | > 0.10 gap signals overfitting |

> **Note**: Accuracy, Precision, Recall, F1, AUC, Gini, Entropy are classification metrics.  
> They are **NOT applicable** to this continuous regression target.

---
## Step 3.1 — Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score

print('Imports complete.')

def root_mean_squared_error(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

ImportError: cannot import name 'root_mean_squared_error' from 'sklearn.metrics' (/home/auyan/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/metrics/__init__.py)

---
## Step 3.2 — Load Preprocessed Data

In [ ]:
X_train_A = np.load('artifacts/X_train_A.npy')
X_test_A  = np.load('artifacts/X_test_A.npy')
X_train_B = np.load('artifacts/X_train_B.npy')
X_test_B  = np.load('artifacts/X_test_B.npy')
y_train   = np.load('artifacts/y_train.npy')
y_test    = np.load('artifacts/y_test.npy')
feature_names_A = joblib.load('artifacts/feature_names_A.pkl')
feature_names_B = joblib.load('artifacts/feature_names_B.pkl')

print(f'X_train_A : {X_train_A.shape}  (scaled, for LR/KNN)')
print(f'X_train_B : {X_train_B.shape}  (unscaled, for DT)')
print(f'y_train   : {y_train.shape},  mean=${y_train.mean():,.0f}')
print(f'y_test    : {y_test.shape},   mean=${y_test.mean():,.0f}')

---
## Step 3.3 — Helper: Evaluate Model

In [ ]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test, cv_splits=5):
    """Train, evaluate, and return metric dict for a regression model."""
    model.fit(X_train, y_train)

    y_pred_train = model.predict(X_train)
    y_pred_test  = model.predict(X_test)

    train_r2   = r2_score(y_train, y_pred_train)
    test_mae   = mean_absolute_error(y_test, y_pred_test)
    test_rmse  = root_mean_squared_error(y_test, y_pred_test)
    test_r2    = r2_score(y_test, y_pred_test)

    kf = KFold(n_splits=cv_splits, shuffle=True, random_state=42)
    cv_rmse_scores = -cross_val_score(
        model, X_train, y_train,
        scoring='neg_root_mean_squared_error', cv=kf, n_jobs=-1
    )
    cv_r2_scores = cross_val_score(
        model, X_train, y_train,
        scoring='r2', cv=kf, n_jobs=-1
    )

    result = {
        'Model'        : name,
        'Train R²'     : round(train_r2, 4),
        'Test MAE'     : round(test_mae, 0),
        'Test RMSE'    : round(test_rmse, 0),
        'Test R²'      : round(test_r2, 4),
        'CV RMSE Mean' : round(cv_rmse_scores.mean(), 0),
        'CV RMSE Std'  : round(cv_rmse_scores.std(), 0),
        'CV R² Mean'   : round(cv_r2_scores.mean(), 4),
        'R² Gap (Overfit?)': round(train_r2 - test_r2, 4),
    }
    print(f'[{name}] MAE=${test_mae:,.0f}  RMSE=${test_rmse:,.0f}  R²={test_r2:.4f}  CV_RMSE=${cv_rmse_scores.mean():,.0f}±{cv_rmse_scores.std():,.0f}')
    return result, model, y_pred_test

---
## Step 3.4 — Model 1: Linear Regression

**Uses Pipeline A** (StandardScaler applied)  
**Expected**: High bias, low variance. Stable but likely underfits non-linear relationships (e.g., location clusters, storey effects).  
**Tuning**: None — Linear Regression has no regularization parameter.

In [ ]:
lr_result, lr_model, lr_pred = evaluate_model(
    'Linear Regression',
    LinearRegression(),
    X_train_A, X_test_A, y_train, y_test
)

In [ ]:
# Top 15 most influential coefficients
coef_df = pd.DataFrame({
    'Feature'    : feature_names_A,
    'Coefficient': lr_model.coef_
}).sort_values('Coefficient', key=abs, ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['steelblue' if c > 0 else 'crimson' for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'][::-1], coef_df['Coefficient'][::-1], color=colors[::-1])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Linear Regression — Top 15 Coefficients (scaled units)', fontsize=12)
ax.set_xlabel('Coefficient Value')
plt.tight_layout()
plt.show()

---
## Step 3.5 — Model 2: KNN Regressor (Default k=5)

**Uses Pipeline A** (StandardScaler is mandatory — KNN is distance-based)  
**Expected**: Low bias with small k (memorizes training), high bias with large k (over-smoothed).  
**Default run** with k=5 first, then full tuning in Segment 5.

In [ ]:
knn_result, knn_model, knn_pred = evaluate_model(
    'KNN Regressor (k=5)',
    KNeighborsRegressor(n_neighbors=5),
    X_train_A, X_test_A, y_train, y_test
)

In [ ]:
# KNN elbow curve — RMSE vs k
k_values = [1, 3, 5, 7, 9, 11, 15, 20, 30]
knn_rmse_train, knn_rmse_test = [], []

for k in k_values:
    m = KNeighborsRegressor(n_neighbors=k)
    m.fit(X_train_A, y_train)
    knn_rmse_train.append(root_mean_squared_error(y_train, m.predict(X_train_A)))
    knn_rmse_test.append(root_mean_squared_error(y_test,  m.predict(X_test_A)))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(k_values, knn_rmse_train, 'b-o', label='Train RMSE')
ax.plot(k_values, knn_rmse_test,  'r-o', label='Test RMSE')
ax.set_xlabel('k (n_neighbors)')
ax.set_ylabel('RMSE (SGD)')
ax.set_title('KNN — RMSE vs k (Elbow Curve)', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

best_k = k_values[np.argmin(knn_rmse_test)]
print(f'Best k on test set: {best_k}  (RMSE=${min(knn_rmse_test):,.0f})')
print('Note: Final k selection will use GridSearchCV with CV in Segment 5.')

---
## Step 3.6 — Model 3: Decision Tree Regressor (Default/Unlimited Depth)

**Uses Pipeline B** (no scaling needed)  
**Criterion**: `squared_error` (minimises within-node variance of resale_price)  
> **Note on Gini/Entropy**: These are used in `DecisionTreeClassifier` for class impurity.  
> For regression, the correct criterion is `squared_error` or `absolute_error`.  

**Expected**: Default (unlimited) tree will perfectly fit training data (R²≈1.0) but overfit significantly on test.

In [ ]:
dt_result, dt_model, dt_pred = evaluate_model(
    'Decision Tree (unlimited depth)',
    DecisionTreeRegressor(criterion='squared_error', random_state=42),
    X_train_B, X_test_B, y_train, y_test
)

In [ ]:
# Decision Tree — depth vs RMSE (learning curve)
depth_values = [2, 3, 4, 5, 6, 7, 8, 10, 12, 15, None]
dt_rmse_train, dt_rmse_test, dt_r2_train, dt_r2_test = [], [], [], []

for depth in depth_values:
    m = DecisionTreeRegressor(criterion='squared_error', max_depth=depth, random_state=42)
    m.fit(X_train_B, y_train)
    dt_rmse_train.append(root_mean_squared_error(y_train, m.predict(X_train_B)))
    dt_rmse_test.append(root_mean_squared_error(y_test,   m.predict(X_test_B)))
    dt_r2_train.append(r2_score(y_train, m.predict(X_train_B)))
    dt_r2_test.append(r2_score(y_test,   m.predict(X_test_B)))

depth_labels = [str(d) if d is not None else 'None' for d in depth_values]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(depth_labels, dt_rmse_train, 'b-o', label='Train RMSE')
axes[0].plot(depth_labels, dt_rmse_test,  'r-o', label='Test RMSE')
axes[0].set_xlabel('max_depth')
axes[0].set_ylabel('RMSE (SGD)')
axes[0].set_title('Decision Tree — RMSE vs max_depth', fontsize=11)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(depth_labels, dt_r2_train, 'b-o', label='Train R²')
axes[1].plot(depth_labels, dt_r2_test,  'r-o', label='Test R²')
axes[1].set_xlabel('max_depth')
axes[1].set_ylabel('R²')
axes[1].set_title('Decision Tree — R² vs max_depth', fontsize=11)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Decision Tree Bias-Variance Tradeoff', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

best_depth_idx = np.argmin(dt_rmse_test)
print(f'Best depth on test RMSE: {depth_labels[best_depth_idx]}  (RMSE=${min(dt_rmse_test):,.0f})')

In [ ]:
# Decision Tree feature importance (depth=8 for interpretable importance)
dt_shallow = DecisionTreeRegressor(criterion='squared_error', max_depth=8, random_state=42)
dt_shallow.fit(X_train_B, y_train)

importance_df = pd.DataFrame({
    'Feature'   : feature_names_B,
    'Importance': dt_shallow.feature_importances_
}).sort_values('Importance', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(importance_df['Feature'][::-1], importance_df['Importance'][::-1], color='steelblue')
ax.set_title('Decision Tree (depth=8) — Top 20 Feature Importances', fontsize=12)
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

---
## Step 3.7 — Baseline Model Comparison Table

In [ ]:
results = pd.DataFrame([lr_result, knn_result, dt_result])
results['Test MAE']     = results['Test MAE'].apply(lambda x: f'${x:,.0f}')
results['Test RMSE']    = results['Test RMSE'].apply(lambda x: f'${x:,.0f}')
results['CV RMSE Mean'] = results['CV RMSE Mean'].apply(lambda x: f'${x:,.0f}')
results['CV RMSE Std']  = results['CV RMSE Std'].apply(lambda x: f'±${x:,.0f}')

print('=== BASELINE MODEL COMPARISON ===')
print(results[['Model','Train R²','Test MAE','Test RMSE','Test R²','CV RMSE Mean','CV RMSE Std','R² Gap (Overfit?)']].to_string(index=False))

---
## Step 3.8 — Actual vs Predicted Scatter Plots

In [ ]:
model_preds = [
    ('Linear Regression', lr_pred),
    ('KNN Regressor',     knn_pred),
    ('Decision Tree',     dt_pred),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, pred) in zip(axes, model_preds):
    ax.scatter(y_test, pred, alpha=0.15, s=3, color='steelblue')
    lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
    ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
    ax.set_xlabel('Actual resale_price')
    ax.set_ylabel('Predicted resale_price')
    ax.set_title(f'{name}\nR²={r2_score(y_test, pred):.4f}', fontsize=11)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
    ax.legend(fontsize=8)

plt.suptitle('Actual vs Predicted — Baseline Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 3.9 — Residual Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, pred) in zip(axes, model_preds):
    residuals = y_test - pred
    ax.scatter(pred, residuals, alpha=0.15, s=3, color='darkorange')
    ax.axhline(0, color='black', linewidth=1.2, linestyle='--')
    ax.set_xlabel('Predicted resale_price')
    ax.set_ylabel('Residual (Actual - Predicted)')
    ax.set_title(f'{name} — Residuals\nMAE=${mean_absolute_error(y_test, pred):,.0f}', fontsize=11)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))

plt.suptitle('Residual Plots — Baseline Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 3.10 — Bias-Variance Interpretation

Run this cell AFTER you see actual metric values to get a data-driven interpretation.

In [ ]:
# Auto-interpret bias-variance behavior
raw_results = [lr_result, knn_result, dt_result]

print('=== BIAS-VARIANCE INTERPRETATION ===')
print()
for r in raw_results:
    name = r['Model']
    gap  = r['R² Gap (Overfit?)']
    test_r2 = r['Test R²']
    print(f'--- {name} ---')
    print(f'  Train R²  : {r["Train R²"]:.4f}')
    print(f'  Test R²   : {test_r2:.4f}')
    print(f'  R² Gap    : {gap:.4f}')
    if test_r2 < 0.70:
        bias_verdict = 'HIGH BIAS (underfitting) — model is too simple for this data'
    elif test_r2 >= 0.90:
        bias_verdict = 'GOOD FIT — strong predictive power'
    else:
        bias_verdict = 'MODERATE FIT — acceptable baseline'
    if gap > 0.15:
        var_verdict = 'HIGH VARIANCE (overfitting) — train-test gap is large'
    elif gap > 0.05:
        var_verdict = 'MODERATE VARIANCE — some overfitting'
    else:
        var_verdict = 'LOW VARIANCE — model generalizes well'
    print(f'  Bias  : {bias_verdict}')
    print(f'  Variance: {var_verdict}')
    print()

---
## Step 3.11 — Save Baseline Results

In [ ]:
import os, json
os.makedirs('artifacts', exist_ok=True)

joblib.dump(lr_model,       'artifacts/model_lr.pkl')
joblib.dump(knn_model,      'artifacts/model_knn_default.pkl')
joblib.dump(dt_model,       'artifacts/model_dt_default.pkl')
joblib.dump(dt_shallow,     'artifacts/model_dt_depth8.pkl')

baseline_df = pd.DataFrame([lr_result, knn_result, dt_result])
baseline_df.to_csv('artifacts/results_baseline.csv', index=False)

print('Saved: model_lr.pkl, model_knn_default.pkl, model_dt_default.pkl, model_dt_depth8.pkl')
print('Saved: results_baseline.csv')

---
## ✅ Segment 3 Complete

**Summary**:
- Linear Regression: establishes linear baseline, likely moderate R² with high bias
- KNN: sensitive to k; elbow curve shows bias-variance tradeoff visually
- Decision Tree: unlimited depth overfits; depth curve shows the optimal depth range

**Next**: Proceed to `04_regularized_boosting.ipynb`
- Ridge, Lasso, Elastic Net (Pipeline A)
- Gradient Boosting Regressor (Pipeline B)
- Explain coefficient shrinkage and ensemble boosting logic